# Ettin classifier (jhu-clsp/ettin-encoder-400m)

### Import Libraries

In [22]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import os
import json

from google.colab import drive, userdata

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report

from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import torch
from datasets import Dataset

### Set up Google Drive mounting and define input/output paths:

In [2]:
# Mount the google drive folder
drive.mount('/content/drive', force_remount = True) # force reconnection

# Define directory, input, and output paths
project_directory = "/content/drive/MyDrive/Colab Notebooks/DS266/final_project"

# Input file with all original-rewrite pairs
final_long_file = os.path.join(project_directory, "data/final_long_w_human.parquet")

# # Output file
# output_file = os.path.join(project_directory, "data/out.parquet")

# Ensure the output directory exists
if not os.path.exists(project_directory):
  os.makedirs(project_directory)

Mounted at /content/drive


### Read the input file:

In [3]:
# Read input file with all original-rewrite pairs
final_long_df = pd.read_parquet(final_long_file)
print('Sucessfully read the input file with all original-rewrite pairs!')

print(f'The input file contains {len(final_long_df)} records/rows.')
print(f'Column names: {list(final_long_df.columns)}')

print('\nThe first 5 rows:')
display(final_long_df.head(3))

Sucessfully read the input file with all original-rewrite pairs!
The input file contains 106120 records/rows.
Column names: ['original', 'rewrite', 'source', 'label']

The first 5 rows:


,original,rewrite,source,label
0,The most prominent composer strongly associate...,The most prominent composer associated with th...,gemini,claude
1,The Persian Wars primarily took place in **mai...,The Persian Wars primarily took place across *...,gemini,claude
2,An **add-on** is something extra or additional...,An **add-on** is an optional component that en...,gemini,claude


In [4]:
# rename columns
final_long_df.rename(columns = {'source': 'original_llm',
                                'label': 'rewrite_llm'},
                     inplace = True)

### Data Exploration & cleaning

In [5]:
print(final_long_df['original_llm'].value_counts(), end = '\n\n')
print(final_long_df['rewrite_llm'].value_counts())

original_llm
gemini    21224
gpt       21224
claude    21224
kimi      21224
human     21224
Name: count, dtype: int64

rewrite_llm
claude    26530
kimi      26530
gemini    26530
gpt       26530
Name: count, dtype: int64


In [6]:
print(final_long_df.isna().sum(), end = '\n\n')

# na_df -- the dataframe that contains nan responses (rewrites)
na_df = final_long_df[final_long_df['rewrite'].isna()]
print(na_df['rewrite_llm'].value_counts())
display(na_df.head(3))
display(na_df.tail(3))

original          0
rewrite         353
original_llm      0
rewrite_llm       0
dtype: int64

rewrite_llm
gpt       347
claude      6
Name: count, dtype: int64


,original,rewrite,original_llm,rewrite_llm
1664,**Machine ethics** is a field of research and ...,None,gemini,claude
2689,"No, your contributions towards Employment Insu...",None,gemini,claude
3805,"The \""Bernanke Twist\"" and \""Operation Twist\""...",None,gemini,claude


,original,rewrite,original_llm,rewrite_llm
103166,"If you're into math, do this thought experimen...",None,human,gpt
103826,It would essentially make goods from other cou...,None,human,gpt
104767,The short of it is that bonds are valued based...,None,human,gpt


In [7]:
# drop nan and duplicates
cleaned_df = final_long_df.dropna(subset = ['rewrite']).drop_duplicates().reset_index(drop = True)
cleaned_df['by_human'] = (cleaned_df['original_llm'] == 'human').astype(int)

print(cleaned_df.isna().sum(), end = '\n\n')
print(cleaned_df['by_human'].value_counts(), end = '\n\n')
display(cleaned_df.head(3))

original        0
rewrite         0
original_llm    0
rewrite_llm     0
by_human        0
dtype: int64

by_human
0    84547
1    21208
Name: count, dtype: int64



,original,rewrite,original_llm,rewrite_llm,by_human
0,The most prominent composer strongly associate...,The most prominent composer associated with th...,gemini,claude,0
1,The Persian Wars primarily took place in **mai...,The Persian Wars primarily took place across *...,gemini,claude,0
2,An **add-on** is something extra or additional...,An **add-on** is an optional component that en...,gemini,claude,0


In [8]:
print(cleaned_df['original_llm'].value_counts(), end = '\n\n')
print(cleaned_df['rewrite_llm'].value_counts(), end = '\n\n')

original_llm
gpt       21220
claude    21214
human     21208
gemini    21089
kimi      21024
Name: count, dtype: int64

rewrite_llm
kimi      26530
gemini    26525
claude    26521
gpt       26179
Name: count, dtype: int64



In [9]:
# stratification

df_human = cleaned_df[cleaned_df['original_llm'] == 'human']
df_gemini = cleaned_df[cleaned_df['original_llm'] == 'gemini']
df_gpt = cleaned_df[cleaned_df['original_llm'] == 'gpt']
df_claude = cleaned_df[cleaned_df['original_llm'] == 'claude']
df_kimi = cleaned_df[cleaned_df['original_llm'] == 'kimi']

df_gemini_sampled = df_gemini.sample(n = int(len(df_human)/4), random_state = 42)
df_gpt_sampled = df_gpt.sample(n = int(len(df_human)/4), random_state = 42)
df_claude_sampled = df_claude.sample(n = int(len(df_human)/4), random_state = 42)
df_kimi_sampled = df_kimi.sample(n = int(len(df_human)/4), random_state = 42)

stratified_df = pd.concat([df_human, df_gemini_sampled, df_gpt_sampled, df_claude_sampled, df_kimi_sampled])
print(stratified_df['original_llm'].value_counts(), end = '\n\n')
print(stratified_df['by_human'].value_counts(), end = '\n\n')
display(stratified_df.head(3))

original_llm
human     21208
gemini     5302
gpt        5302
claude     5302
kimi       5302
Name: count, dtype: int64

by_human
1    21208
0    21208
Name: count, dtype: int64



,original,rewrite,original_llm,rewrite_llm,by_human
21218,"Composers and works include Barbara Kolb , Pau...",Composers employing sound mass techniques incl...,human,claude,1
21219,The Greco-Persian Wars (also often called the ...,The Greco-Persian Wars were a series of confli...,human,claude,1
21220,"Plug-in (computing) , a piece of software whic...",# Extensions and Enhancements for Software and...,human,claude,1


In [10]:
# Shuffle the data
shuffled_indices = np.random.permutation(len(stratified_df))
shuffled_df = stratified_df.iloc[shuffled_indices].reset_index(drop = True)
display(shuffled_df.head(3))

,original,rewrite,original_llm,rewrite_llm,by_human
0,"I can’t diagnose over chat, but the signs you ...","While I cannot provide a diagnosis, the sympto...",gpt,gemini,0
1,Below is a plain-English checklist of the main...,Tax-smart Savings & Investing Checklist \n(Pl...,kimi,kimi,0
2,Short answer: Yes — but it depends what you me...,**Can I get a virtual terminal without a separ...,gpt,gemini,0


In [11]:
# train-test split

train_df, test_df = train_test_split(shuffled_df, test_size = 0.2, stratify = shuffled_df['by_human'], random_state = 42)
print(f'Total samples: {len(shuffled_df)}')
print(f'Training samples: {len(train_df)}')
print(f'Testing samples: {len(test_df)}')

Total samples: 42416
Training samples: 33932
Testing samples: 8484


In [24]:
# Convert the training and testing dataset to Dataset format

train_data = Dataset.from_pandas(train_df[['original', 'rewrite', 'by_human']]).rename_column('by_human', 'labels')
test_data = Dataset.from_pandas(test_df[['original', 'rewrite', 'by_human']]).rename_column('by_human', 'labels')

### Ettin classifier development

Define tokenizer

In [25]:
ettin_checkpoint = 'jhu-clsp/ettin-encoder-400m'
ettin_tokenizer = AutoTokenizer.from_pretrained(ettin_checkpoint)

Define the training and testing inputs

In [26]:
def tokenization(df):
  return ettin_tokenizer(df['original'],
                         df['rewrite'],
                         padding = 'max_length',
                         truncation = True,
                         max_length = 512)

train_tokenized = train_data.map(tokenization, batched = True)
train_tokenized.set_format('torch')

test_tokenized = test_data.map(tokenization, batched = True)
test_tokenized.set_format('torch')

Map:   0%|          | 0/33932 [00:00<?, ? examples/s]

Map:   0%|          | 0/8484 [00:00<?, ? examples/s]

Define metrics

In [27]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis = -1)

    return {'accuracy': accuracy_score(labels, predictions),
            'f1_score': f1_score(labels, predictions, average = 'binary')}

### Baseline model (untrained Ettin)

In [28]:
ettin_baseline_model = AutoModelForSequenceClassification.from_pretrained(ettin_checkpoint, num_labels = 2) # binary classification

baseline_training_args = TrainingArguments(output_dir = './ettin_baseline',
                                           per_device_eval_batch_size = 8)

baseline_trainer = Trainer(model = ettin_baseline_model,
                           args = baseline_training_args,
                           eval_dataset = test_tokenized,
                           compute_metrics = compute_metrics)

Loading weights:   0%|          | 0/172 [00:00<?, ?it/s]

ModernBertForSequenceClassification LOAD REPORT from: jhu-clsp/ettin-encoder-400m
Key               | Status     | 
------------------+------------+-
decoder.weight    | UNEXPECTED | 
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
baseline_results = baseline_trainer.evaluate()

print(f"(Baseline) Accuracy: {baseline_results['accuracy']} | F1: {baseline_results['eval_f1']}")

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


### Fine-tuned model (trained)

In [ ]:
ettin_finetuned_model = AutoModelForSequenceClassification.from_pretrained(ettin_checkpoint, num_labels = 2) # binary classification

finetune_training_args = TrainingArguments(output_dir = './ettin_finetuned',
                                           num_train_epochs = 3, # likely to converge already
                                           learning_rate = 0.00002,
                                           per_device_train_batch_size = 8,   # to prevent overfitting
                                           per_device_eval_batch_size = 8,   # to prevent overfitting
                                           evaluation_strategy = 'epoch',
                                           save_strategy = 'epoch',
                                           weight_decay = 0.01, # prevent overfitting to specific punctuation habits
                                           load_best_model_at_end = True,
                                           metric_for_best_model = 'f1',
                                           logging_steps = 100,
                                           warmup_steps = 500) # warmup steps = 500 to stablize gradients

finetune_trainer = Trainer(model = ettin_finetuned_model,
                           args = finetune_training_args,
                           train_dataset = train_tokenized,
                           eval_dataset = test_tokenized,
                           compute_metrics = compute_metrics)

finetune_trainer.train()

In [ ]:
finetuned_results = finetune_trainer.evaluate()

print(f"(Baseline) Accuracy: {baseline_results['accuracy']} | F1: {baseline_results['eval_f1']}")
print(f"(Finetuned) Accuracy: {finetuned_results['accuracy']} | F1: {finetuned_results['eval_f1']}")

### Visualization

In [ ]:
predictions = finetune_trainer.predict(test_data)
y_pred = np.argmax(predictions.predictions, axis=-1)
y_true = test_df['by_human'].values

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Machine', 'Human'],
            yticklabels=['Machine', 'Human'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Ettin Binary Classifier Confusion Matrix')
plt.show()

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=['Machine', 'Human']))

### Save the final dataframe to parquet

In [ ]:
finetune_trainer.save_model("./finetuned_ettin")